# Tools in LangChain

LangChain is a popular framework for building applications with language models (like GPT) that can interact with other data sources and perform complex tasks. In LangChain, “tools” are components that allow an agent (a model that can take multiple actions) to perform specific tasks beyond just generating text. Essentially, a tool is something the agent can “call” to get information, perform actions, or manipulate data.

## Install Libraries

In [ ]:
!pip install langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental -U ddgs

# Import Libraries

In [4]:
from langchain_community.tools import tool, DuckDuckGoSearchRun

In [13]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [19]:
from langchain.tools import BaseTool
from typing import Type

# Built in Tools

- **References**: https://docs.langchain.com/oss/python/integrations/tools

## 1. Duck Duck Go

In [2]:
search_engine = DuckDuckGoSearchRun()

search_engine.invoke('What is Brave?')

'3 days ago - Brave is a free and open-source web browser which was first released in 2016. It is developed by US-based Brave Software, Inc. and based on the Chromium web browser. The browser is marketed as a privacy-focused web browser and includes features such as built-in advertisement blocking, protections ... September 6, 2025 - Atlanta Braves, an American Major League Baseball team (originally the Boston Braves, then the Milwaukee Braves), or their affiliates: October 16, 2025 - Brave Search is the world’s most complete, independent, private search engine. By integrating Brave Search into its browser, Brave offers the first all-in-one browser / search alternative to the big tech platforms. How do I install Brave on Linux using the terminal? What are the system requirements to install Brave ? Changes to macOS desktop browser requirements. See all 12 articles. Today Brave is releasing the Brave Search API, allowing developers to incorporate high-quality search results from billions

In [3]:
print(search_engine.name)
print(search_engine.description)
print(search_engine.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# Custom Tools Build

# Method 1: @tool decorator

In [5]:
@tool
def multiply(num1: int, num2: int)->int:
    """This is a multiplication function which takes two integers argument num1 and num2 and return a single integer of two numbers"""
    return num1*num2

In [6]:
result = multiply.invoke({
    'num1' : 5,
    'num2' : 3
})

result

15

In [8]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
This is a multiplication function which takes two integers argument num1 and num2 and return a single integer of two numbers
{'num1': {'title': 'Num1', 'type': 'integer'}, 'num2': {'title': 'Num2', 'type': 'integer'}}


In [10]:
multiply.args_schema.model_json_schema()

{'description': 'This is a multiplication function which takes two integers argument num1 and num2 and return a single integer of two numbers',
 'properties': {'num1': {'title': 'Num1', 'type': 'integer'},
  'num2': {'title': 'Num2', 'type': 'integer'}},
 'required': ['num1', 'num2'],
 'title': 'multiply',
 'type': 'object'}

# Method 2 - Using StructuredTool

In [15]:
class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to multiply")
    b: int = Field(description="The second number to multiply")

In [16]:
def multiply_fn(a, b):
    return a*b

In [17]:
multiply_tool = StructuredTool.from_function(
    func=multiply_fn,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [18]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


# Method 3 - Using BaseTool Class

In [21]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(description="The first number to add")
    b: int = Field(description="The second number to add")

In [22]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [23]:
multiply_tool = MultiplyTool()

result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


# Toolkit (Multiple Tools)

In [24]:
# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [25]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [26]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for tool in tools:
    print(tool.name, "=>", tool.description)

add => Add two numbers
multiply => Multiply two numbers
